# Homework 1: Agentic RAG

Solution for [LLM Zoomcamp 2026 - Homework 1](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/homework/01-agentic-rag/homework.md).

We build a RAG system from scratch over the **course lessons** (instead of the FAQ), then turn it into an agent.



> LLM-based answers (Q3, Q5, Q6) can vary slightly between runs - the homework says to pick the closest option.

## Setup

Dependencies (already in `pyproject.toml`): `minsearch`, `openai`, `python-dotenv`, `gitsource`, `toyaikit`.

The OpenAI API key is read from the `.env` file at the repo root (`OPENAI_API_KEY=...`).

In [42]:
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY from .env

True

## Preparation

Pull the lesson pages straight from the course repository, pinned to commit `8c1834d` so everyone works with the same data.

- `allowed_extensions={"md"}` - only look at markdown files
- `filename_filter` - keep only files under a module's `lessons/` folder (skips READMEs etc.)

In [43]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

Each file's `parse()` returns a dict with `filename` and `content`:

In [44]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

A quick look at the first document:

In [45]:
print(documents[0]['content'])

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 

## Q1. How many lesson pages

> How many lesson pages are in the dataset?
>
> Options: 24 · 72 · 240 · 720

In [46]:
print(f"There are total {len(documents)} pages in the dataset")

There are total 72 pages in the dataset


**Answer: 72**

## Q2. Indexing and searching

> Index the documents with minsearch - make `content` a text field and `filename` a keyword field. Then search with this query:
>
> *How does the agentic loop keep calling the model until it stops?*
>
> What's the `filename` of the first result?
>
> Options: `01-agentic-rag/lessons/03-rag.md` · `01-agentic-rag/lessons/14-agentic-loop.md` · `04-evaluation/lessons/13-llm-as-judge.md` · `06-best-practices/lessons/02-hybrid-search.md`

In [47]:
from minsearch import Index


def build_index(documents):
    """Build a minsearch index: full-text search on `content`, exact-match filter on `filename`."""
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index


index = build_index(documents)

In [48]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(question, num_results=5)

# Show only the filenames - the full results contain entire lesson pages
[doc["filename"] for doc in search_results]

['01-agentic-rag/lessons/14-agentic-loop.md',
 '01-agentic-rag/lessons/15-frameworks.md',
 '01-agentic-rag/lessons/13-function-calling.md',
 '01-agentic-rag/lessons/11-agents-intro.md',
 '01-agentic-rag/lessons/16-other-frameworks.md']

In [49]:
print(search_results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


**Answer: `cohorts/2026/01-agentic-rag/lessons/14-agentic-loop.md`**

(At commit `8c1834d` the paths don't have the `cohorts/2026/` prefix yet, but it's the same file.)

## Q3. RAG

> Build a RAG over the index from Q2 and answer the query: *How does the agentic loop keep calling the model until it stops?*
>
> Use `gpt-5.4-mini`. How many input (prompt) tokens did we send to the model for this request?
>
> Options: 700 · 7000 · 70000 · 700000

`RAGBase` from the lessons was written for the FAQ schema (`section` / `question` / `answer`). Our documents have `filename` / `content`, so I adapted [`rag_helper.py`](rag_helper.py):

- `search` - removed the FAQ-specific boosting and `course` filter
- `build_context` - uses `filename` and `content`
- `llm` - returns the whole response instead of just the text
- `rag` - returns `(answer, input_tokens)`, read from `response.usage.input_tokens`

Original helper:
```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/01-agentic-rag/code/rag_helper.py
```

In [50]:
from openai import OpenAI

from rag_helper import RAGBase  # local, adapted version (see rag_helper.py)

openai_client = OpenAI()

In [51]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [52]:
answer, input_tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [53]:
print(answer)

It keeps calling the model inside a `while True` loop.

Each iteration:
1. Send the full message history to the model.
2. Check the response for any `function_call` items.
3. If there are tool calls, run them, append the results to memory, and loop again.
4. If there are no function calls, `break` out of the loop.

So the stop condition is: **no function calls in the current response**.


In [54]:
print(input_tokens)

7136


**Answer: 7000** (measured: 7136 input tokens)

## Q4. Chunking

> Long documents make retrieval less precise. Split each page into smaller, overlapping pieces with `chunk_documents` (sliding window):
>
> ```python
> chunks = chunk_documents(documents, size=2000, step=1000)
> ```
>
> - each chunk is a window of `size` characters
> - the window moves forward by `step` characters, so consecutive chunks overlap by `size - step` = 1000 characters
> - every chunk keeps `filename` and adds `start` (offset in the page) and `content` (chunk text)
>
> How many chunks do you get?
>
> Options: 70 · 295 · 1100 · 4500

In [55]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [56]:
print(len(chunks))

295


A chunk looks like this:

In [57]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

**Answer: 295**

## Q5. RAG with chunking

> Index the chunks from Q4 (same fields as before), point your RAG at the chunk index, and answer the same query again.
>
> Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
>
> Options: about the same · 3× fewer · 10× fewer · 30× fewer

In [58]:
index_chunks = build_index(chunks)

assistant_chunk = RAGBase(
    index=index_chunks,
    llm_client=openai_client,
)

answer_chunk, input_tokens_chunk = assistant_chunk.rag("How does the agentic loop keep calling the model until it stops?")

In [59]:
print("Response:", answer_chunk)
print()
print("Input tokens:", input_tokens_chunk)

Response: It keeps calling the model inside a `while True` loop. After each response, the code checks whether the model returned any `function_call` items:

- If there is at least one function call, it runs the tool, appends the result to `messages`, and loops again.
- If there are no function calls, it breaks out of the loop.

So the stopping condition is: **no function calls in the latest response**.

Input tokens: 2319


In [60]:
# Ratio of input tokens: full pages (Q3) vs chunks
7136 / 2319

3.0771884432945233

**Answer: 3× fewer** (7136 / 2319 ≈ 3.1)

## Q6. Turning it into an agent

> Make it agentic: give the LLM a `search` tool (over the chunk index) and let it decide when and what to search. Use these instructions:
>
> *You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.*
>
> Ask it: *How does the agentic loop work, and how is it different from plain RAG?*
>
> How many times did the agent call `search`?
>
> Options: 0 · 4 · 10 · 20

I used [toyaikit](https://github.com/alexeygrigorev/toyaikit). It builds the tool schema from the function's type hints and docstring.

In [61]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [62]:
def search(query: str) -> list[dict]:
    """
    Search the lessons database for entries matching the given query.

    """
    return index_chunks.search(query, num_results=5)

In [63]:
agent_tools = Tools()
agent_tools.add_tool(search)

The tool schema toyaikit generated from the function:

In [64]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [65]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

instructions = (
    "You're a course teaching assistant. Answer the student's question using the search tool. "
    "Make multiple searches with different keywords before answering."
)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

In [66]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


Cost of the run and the full message history (tool calls, tool outputs and the final answer):

In [67]:
result.cost

CostInfo(input_cost=Decimal('0.006087'), output_cost=Decimal('0.002664'), total_cost=Decimal('0.008751'))

In [68]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"agentic loop RAG"}', call_id='call_j4p8KPSt4IqBDhYXDYfK4By9', name='search', type='function_call', id='fc_04b976013dd6e4e3006ad3216a4ddc87d1982fa3a267c85450', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"plain RAG versus agentic loop"}', call_id='call_54ED9vEPauT9p6T9TKeYVVlw', name='search', type='function_call', id='fc_04b976013dd6e4e3006ad3216a4dec87d1a76f88b5399e81b7', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"agentic workflow retrieval generation l

Count the `search` tool calls in the message history:

In [69]:
search_calls = [
    m for m in result.all_messages
    if getattr(m, "type", None) == "function_call" and m.name == "search"
]

for call in search_calls:
    print(call.arguments)

print(f"\nThe agent called search {len(search_calls)} times")

{"query":"agentic loop RAG"}
{"query":"plain RAG versus agentic loop"}
{"query":"agentic workflow retrieval generation lesson"}

The agent called search 3 times


**Answer: 4** (measured: 3 calls - the closest option is 4)

The agent made all 3 searches in its first turn, then answered from the results without searching again.